# Week 3: Feature Engineering

## Objective
This phase focuses on creating time-based and lag features from the hourly
energy consumption dataset to prepare inputs for baseline and advanced
machine learning models.

In [1]:
import pandas as pd
import numpy as np

In [3]:
hourly_df = pd.read_csv(
    "../../data/processed/hourly_device_energy.csv",
    parse_dates=['timestamp']
)

hourly_df.head()

,Appliance Type,timestamp,Energy Consumption (kWh)
0,Air Conditioning,2023-01-01 00:00:00,4.42
1,Air Conditioning,2023-01-01 01:00:00,7.61
2,Air Conditioning,2023-01-01 02:00:00,14.80
3,Air Conditioning,2023-01-01 03:00:00,4.19
4,Air Conditioning,2023-01-01 04:00:00,10.68


In [4]:
hourly_df = hourly_df.set_index('timestamp')

In [5]:
hourly_df['hour'] = hourly_df.index.hour
hourly_df['day'] = hourly_df.index.day
hourly_df['weekday'] = hourly_df.index.weekday
hourly_df['month'] = hourly_df.index.month

hourly_df.head()

,Appliance Type,Energy Consumption (kWh),hour,day,weekday,month
timestamp,,,,,,
2023-01-01 00:00:00,Air Conditioning,4.42,0,1,6,1
2023-01-01 01:00:00,Air Conditioning,7.61,1,1,6,1
2023-01-01 02:00:00,Air Conditioning,14.80,2,1,6,1
2023-01-01 03:00:00,Air Conditioning,4.19,3,1,6,1
2023-01-01 04:00:00,Air Conditioning,10.68,4,1,6,1


In [6]:
hourly_df['lag_1'] = hourly_df['Energy Consumption (kWh)'].shift(1)
hourly_df['lag_24'] = hourly_df['Energy Consumption (kWh)'].shift(24)

hourly_df['rolling_mean_24'] = (
    hourly_df['Energy Consumption (kWh)']
    .rolling(window=24)
    .mean()
)

hourly_df.head(30)

,Appliance Type,Energy Consumption (kWh),hour,day,weekday,month,lag_1,lag_24,rolling_mean_24
timestamp,,,,,,,,,
2023-01-01 00:00:00,Air Conditioning,4.42,0,1,6,1,NaN,NaN,NaN
2023-01-01 01:00:00,Air Conditioning,7.61,1,1,6,1,4.42,NaN,NaN
2023-01-01 02:00:00,Air Conditioning,14.80,2,1,6,1,7.61,NaN,NaN
2023-01-01 03:00:00,Air Conditioning,4.19,3,1,6,1,14.80,NaN,NaN
2023-01-01 04:00:00,Air Conditioning,10.68,4,1,6,1,4.19,NaN,NaN
2023-01-01 05:00:00,Air Conditioning,6.94,5,1,6,1,10.68,NaN,NaN
2023-01-01 06:00:00,Air Conditioning,0.00,6,1,6,1,6.94,NaN,NaN
2023-01-01 07:00:00,Air Conditioning,0.00,7,1,6,1,0.00,NaN,NaN
2023-01-01 08:00:00,Air Conditioning,0.00,8,1,6,1,0.00,NaN,NaN


In [7]:
hourly_df = hourly_df.dropna()
hourly_df.shape

(87804, 9)

In [9]:
appliances = hourly_df['Appliance Type'].unique()

for appliance in appliances:
    appliance_df = hourly_df[
        hourly_df['Appliance Type'] == appliance
    ].copy()

    X = appliance_df.drop(
        ['Energy Consumption (kWh)', 'Appliance Type'],
        axis=1
    )
    y = appliance_df['Energy Consumption (kWh)']

    split_index = int(len(appliance_df) * 0.8)

    X_train = X.iloc[:split_index]
    X_test = X.iloc[split_index:]
    y_train = y.iloc[:split_index]
    y_test = y.iloc[split_index:]

    X_train.to_csv(f"../../data/model_inputs/X_train_{appliance}.csv", index=False)
    X_test.to_csv(f"../../data/model_inputs/X_test_{appliance}.csv", index=False)
    y_train.to_csv(f"../../data/model_inputs/y_train_{appliance}.csv", index=False)
    y_test.to_csv(f"../../data/model_inputs/y_test_{appliance}.csv", index=False)

print("Train-test datasets saved in data/model_inputs/")


Train-test datasets saved in data/model_inputs/
